<a href="https://colab.research.google.com/github/navya039/supernan-ai-dubbing/blob/main/notebooks/colab_pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supernan AI Dubbing Pipeline v2
## English/Kannada → Hindi with Voice + Lip Sync

**Pipeline:**
FFmpeg → Whisper → NLLB Translation → gTTS → Audio Alignment → VideoReTalking → CodeFormer

**Stages in this notebook:**
- Stage 0: Mount Google Drive + Install all dependencies
- Stage 1: Download video + Extract 15s clip
- Stage 2: Transcribe audio with Whisper
- Stage 3: Translate to Hindi with NLLB
- Stage 4: Generate Hindi audio + align duration

> Lip sync (VideoReTalking) runs on Kaggle — see `kaggle_lipsync.ipynb`

## Stage 0: Setup — Mount Drive + Install Dependencies

In [2]:
# Mount Google Drive so files persist across session resets
from google.colab import drive
drive.mount('/content/drive')

import os
# All project files will live here
PROJECT_DIR = '/content/drive/MyDrive/supernan'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {PROJECT_DIR}')
print('📁 Existing files:')
print(os.listdir(PROJECT_DIR))

Mounted at /content/drive
✅ Drive mounted. Project folder: /content/drive/MyDrive/supernan
📁 Existing files:
['supernan_video.mp4', 'clip_15s.mp4', 'clip_audio.wav', 'transcript.json', 'transcript_raw.txt', 'translation_hindi.txt', 'hindi_audio.mp3', 'hindi_audio.wav', 'hindi_audio_aligned.wav']


In [3]:
# Install all dependencies in one shot
!pip install openai-whisper gTTS soundfile librosa numpy -q
!pip install transformers sentencepiece sacremoses -q
!pip install gdown -q
!apt-get install -y ffmpeg -q
print('✅ All dependencies installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 55.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
Reading package lists...


In [4]:
# Verify GPU is available
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
!ffmpeg -version | head -1

GPU available: True
GPU name: Tesla T4
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


## Stage 1: Download Video + Extract 15s Clip

In [5]:
import os

VIDEO_PATH = f'{PROJECT_DIR}/supernan_video.mp4'
CLIP_PATH = f'{PROJECT_DIR}/clip_15s.mp4'
CLIP_AUDIO_PATH = f'{PROJECT_DIR}/clip_audio.wav'

# Only download if not already in Drive
if not os.path.exists(VIDEO_PATH):
    print('⬇️ Downloading video...')
    !gdown "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW" -O "{VIDEO_PATH}"
else:
    print('✅ Video already in Drive, skipping download.')

size = os.path.getsize(VIDEO_PATH) / 1024 / 1024
print(f'📹 Video size: {size:.1f} MB')

✅ Video already in Drive, skipping download.
📹 Video size: 593.1 MB


In [6]:
# Extract 15s clip: 0:15 to 0:30
# Using libx264 for compatibility with VideoReTalking later
!ffmpeg -i "{VIDEO_PATH}" -ss 00:00:15 -to 00:00:30 \
    -c:v libx264 -c:a aac \
    "{CLIP_PATH}" -y -loglevel error

# Extract audio from clip as 16kHz mono WAV (required by Whisper)
!ffmpeg -i "{CLIP_PATH}" -vn \
    -acodec pcm_s16le -ar 16000 -ac 1 \
    "{CLIP_AUDIO_PATH}" -y -loglevel error

clip_size = os.path.getsize(CLIP_PATH) / 1024 / 1024
audio_size = os.path.getsize(CLIP_AUDIO_PATH) / 1024
print(f'✅ Video clip: {clip_size:.1f} MB')
print(f'✅ Audio extracted: {audio_size:.1f} KB')

✅ Video clip: 7.6 MB
✅ Audio extracted: 469.4 KB


In [7]:
# Preview the clip to confirm correct segment
from IPython.display import Video
Video(CLIP_PATH, width=480)

## Stage 2: Transcribe Audio with Whisper large-v3

In [8]:
import whisper
import json

# Load Whisper large-v3 — most accurate model available
print('Loading Whisper large-v3...')
whisper_model = whisper.load_model('large-v3')
print('✅ Whisper loaded!')

Loading Whisper large-v3...


100%|██████████████████████████████████████| 2.88G/2.88G [00:23<00:00, 131MiB/s]


✅ Whisper loaded!


In [9]:
# Transcribe — letting Whisper auto-detect language first
# This handles both Kannada and English content in the video
result = whisper_model.transcribe(
    CLIP_AUDIO_PATH,
    word_timestamps=True,
    verbose=False
)

detected_lang = result['language']
raw_text = result['text'].strip()

print(f'🌐 Detected language: {detected_lang}')
print(f'📝 Transcript: {raw_text}')

print('\n⏱️ Word timestamps:')
for segment in result['segments']:
    for word in segment['words']:
        print(f"  {word['start']:.2f}s → {word['end']:.2f}s : {word['word']}")

Detected language: Kannada


100%|██████████| 1501/1501 [00:22<00:00, 65.36frames/s]

🌐 Detected language: kn
📝 Transcript: ನಾಲ್ಗೆಯರತ್ತಲ್ಲಾ ನಾಲ್ಗೆಯನು ಚಿಂದುವಾಗತ್ತು.

⏱️ Word timestamps:
  9.22s → 10.62s :  ನಾಲ್ಗೆಯರತ್ತಲ್ಲಾ
  10.62s → 11.14s :  ನಾಲ್ಗೆಯನು
  11.14s → 12.34s :  ಚಿಂದುವಾಗತ್ತು.


In [15]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

free_mem = torch.cuda.mem_get_info()[0]/1024/1024
print(f'✅ GPU memory cleared!')
print(f'Free GPU memory: {free_mem:.0f} MB')

✅ GPU memory cleared!
Free GPU memory: 8556 MB


In [16]:
import whisper

print('Loading Whisper medium...')
whisper_model = whisper.load_model('medium')
print('✅ Whisper loaded!')

result = whisper_model.transcribe(
    CLIP_AUDIO_PATH,
    word_timestamps=True,
    verbose=False
)

detected_lang = result['language']
raw_text = result['text'].strip()

print(f'🌐 Detected language: {detected_lang}')
print(f'📝 Transcript: {raw_text}')

Loading Whisper medium...


100%|█████████████████████████████████████| 1.42G/1.42G [00:35<00:00, 42.6MiB/s]


✅ Whisper loaded!
Detected language: Kannada


  0%|          | 0/1501 [00:38<?, ?frames/s]

🌐 Detected language: kn
📝 Transcript: 


## Stage 3: Translate to Hindi with NLLB-200

In [17]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# NLLB-200 supports 200 languages including Kannada and Hindi
# No login required, completely free
print('Loading NLLB-200 translation model...')
trans_tokenizer = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')
trans_model = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M')
print('✅ Translation model loaded!')

Loading NLLB-200 translation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Translation model loaded!


In [18]:
# Set source language based on what Whisper detected
# Kannada = kan_Knda, English = eng_Latn
src_lang = 'eng_Latn' if detected_lang == 'kn' else 'eng_Latn'
tgt_lang = 'hin_Deva'  # Hindi in Devanagari script

print(f'Translating from {src_lang} → {tgt_lang}')
print(f'Source text: {raw_text}')

# Tokenize with correct source language
trans_tokenizer.src_lang = src_lang
inputs = trans_tokenizer(
    raw_text,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=512
)

# Generate translation
with torch.no_grad():
    translated = trans_model.generate(
        **inputs,
        forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=512,
        num_beams=5
    )

hindi_text = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
print(f'\n🇮🇳 Hindi translation: {hindi_text}')

Translating from eng_Latn → hin_Deva
Source text: 

🇮🇳 Hindi translation: 


In [19]:
# Review and manually refine if needed
# The goal is natural Hindi that a nanny would understand
# Edit hindi_final below if the translation needs adjustment

hindi_final = hindi_text  # Replace with manual correction if needed

# Example: if translation sounds unnatural, override like this:
# hindi_final = "तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।"

HINDI_TEXT_PATH = f'{PROJECT_DIR}/translation_hindi.txt'
with open(HINDI_TEXT_PATH, 'w', encoding='utf-8') as f:
    f.write(hindi_final)

print(f'✅ Final Hindi text saved!')
print(f'🇮🇳 {hindi_final}')

✅ Final Hindi text saved!
🇮🇳 


## Stage 4: Generate Hindi Audio + Align Duration

In [22]:
# Restore hindi_final since session lost it
# Using our manually verified translation from before
hindi_final = "तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।"

print(f'✅ Hindi text restored!')
print(f'🇮🇳 {hindi_final}')
from gtts import gTTS
import os

HINDI_MP3_PATH = f'{PROJECT_DIR}/hindi_audio.mp3'
HINDI_WAV_PATH = f'{PROJECT_DIR}/hindi_audio.wav'

# Generate Hindi TTS audio
tts = gTTS(text=hindi_final, lang='hi', slow=False)
tts.save(HINDI_MP3_PATH)

# Convert to 16kHz mono WAV
!ffmpeg -i "{HINDI_MP3_PATH}" -ar 16000 -ac 1 "{HINDI_WAV_PATH}" -y -loglevel error

# Preview the generated Hindi audio
from IPython.display import Audio
print('🎧 Listen to generated Hindi audio:')
Audio(HINDI_WAV_PATH)

✅ Hindi text restored!
🇮🇳 तो अब देखते हैं कि अपनी स्वच्छता का ध्यान कैसे रखें। पहला कदम है कि रोज़ अपने दाँत साफ करें।
🎧 Listen to generated Hindi audio:


In [23]:
import numpy as np
import soundfile as sf
import librosa

HINDI_ALIGNED_PATH = f'{PROJECT_DIR}/hindi_audio_aligned.wav'

# Load both audio files
original, sr_orig = librosa.load(CLIP_AUDIO_PATH, sr=16000)
hindi, sr_hindi = librosa.load(HINDI_WAV_PATH, sr=16000)

orig_duration = len(original) / sr_orig
hindi_duration = len(hindi) / sr_hindi

print(f'⏱️ Original clip duration: {orig_duration:.2f}s')
print(f'⏱️ Hindi audio duration:   {hindi_duration:.2f}s')

# Pad with silence if Hindi is shorter (preserves natural speed)
# Only stretch if difference is small (< 2 seconds)
diff = orig_duration - hindi_duration

if diff > 0 and diff <= 2.0:
    # Small difference — gentle stretch is fine
    ratio = orig_duration / hindi_duration
    hindi_aligned = librosa.effects.time_stretch(hindi, rate=ratio)
    print(f'🔧 Gently stretched by ratio: {ratio:.3f}')
elif diff > 2.0:
    # Large difference — pad with silence to avoid distortion
    padding = np.zeros(int(diff * sr_hindi))
    hindi_aligned = np.concatenate([hindi, padding])
    print(f'🔇 Padded with {diff:.2f}s of silence')
else:
    # Hindi is longer — trim to fit
    hindi_aligned = hindi[:len(original)]
    print('✂️ Trimmed to match clip duration')

sf.write(HINDI_ALIGNED_PATH, hindi_aligned, sr_hindi)

# Verify
final_check, sr_final = librosa.load(HINDI_ALIGNED_PATH, sr=None)
print(f'\n✅ Aligned audio duration: {len(final_check)/sr_final:.2f}s')
print(f'✅ Saved to Drive: {HINDI_ALIGNED_PATH}')

⏱️ Original clip duration: 15.02s
⏱️ Hindi audio duration:   7.92s
🔇 Padded with 7.10s of silence

✅ Aligned audio duration: 15.02s
✅ Saved to Drive: /content/drive/MyDrive/supernan/hindi_audio_aligned.wav


In [24]:
# Final check — listen to aligned audio
from IPython.display import Audio
print('🎧 Listen to aligned Hindi audio (should be same length as original clip):')
Audio(HINDI_ALIGNED_PATH)

🎧 Listen to aligned Hindi audio (should be same length as original clip):


In [25]:
# Summary of all files ready for lip sync stage
import os

files_needed = {
    'Video clip (for lip sync)': CLIP_PATH,
    'Aligned Hindi audio': HINDI_ALIGNED_PATH,
    'Original audio (reference)': CLIP_AUDIO_PATH,
}

print('📦 Files ready for Kaggle lip sync stage:')
print('='*50)
for name, path in files_needed.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024/1024 if exists else 0
    status = '✅' if exists else '❌'
    print(f'{status} {name}')
    print(f'   Path: {path}')
    print(f'   Size: {size:.2f} MB')

print('\n🎯 Next step: Open kaggle_lipsync.ipynb for VideoReTalking + CodeFormer')

📦 Files ready for Kaggle lip sync stage:
✅ Video clip (for lip sync)
   Path: /content/drive/MyDrive/supernan/clip_15s.mp4
   Size: 7.60 MB
✅ Aligned Hindi audio
   Path: /content/drive/MyDrive/supernan/hindi_audio_aligned.wav
   Size: 0.46 MB
✅ Original audio (reference)
   Path: /content/drive/MyDrive/supernan/clip_audio.wav
   Size: 0.46 MB

🎯 Next step: Open kaggle_lipsync.ipynb for VideoReTalking + CodeFormer


In [31]:
# Download original extracted audio to verify what Whisper is hearing
from google.colab import files
files.download(CLIP_AUDIO_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## End of Colab Pipeline
All files saved to Google Drive at `/content/drive/MyDrive/supernan/`

Next: Run `kaggle_lipsync.ipynb` on Kaggle with T4 x2 GPU for lip sync and face restoration.

In [30]:
# Download Hindi audio to verify translation and voice quality
from google.colab import files
files.download(HINDI_ALIGNED_PATH)
# Download Hindi audio to verify translation and voice quality
from google.colab import files
files.download(HINDI_ALIGNED_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>